# Prepare Calendar Table

Builds the standard calendar dimension `databricks_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
# Calendar range and target table
start_date = "1940-01-01"
end_date = "2030-12-31"
target_table = "databricks_caresync.silver.calendar"

spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Standard calendar dimension, one row per day from {start_date} to {end_date}'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'{start_date}', DATE'{end_date}', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)              AS datekey,
  date,
  -- Year
  year(date)                                              AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  -- Quarter
  quarter(date)                                           AS quarter,
  concat('Q', quarter(date))                              AS quarter_name,
  concat(year(date), '-Q', quarter(date))                 AS year_quarter,
  CAST(date_trunc('QUARTER', date) AS DATE)               AS quarter_start_date,
  CAST(add_months(date_trunc('QUARTER', date), 3) AS DATE) - 1 AS quarter_end_date,
  datediff(date, CAST(date_trunc('QUARTER', date) AS DATE)) + 1 AS day_of_quarter,
  -- Month
  month(date)                                             AS month,
  date_format(date, 'MMMM')                               AS month_name,
  date_format(date, 'MMM')                                AS month_short_name,
  date_format(date, 'yyyy-MM')                            AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                AS year_month_key,
  CAST(trunc(date, 'MM') AS DATE)                         AS month_start_date,
  last_day(date)                                          AS month_end_date,
  day(last_day(date))                                     AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(date)                                        AS week,
  extract(YEAROFWEEK FROM date)                           AS iso_year,
  concat(extract(YEAROFWEEK FROM date), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  date_sub(date, weekday(date))                           AS week_start_date,
  date_add(date, 6 - weekday(date))                       AS week_end_date,
  CAST(ceil(day(date) / 7.0) AS INT)                      AS week_of_month,
  -- Day
  day(date)                                               AS day_of_month,
  dayofyear(date)                                         AS day_of_year,
  weekday(date) + 1                                       AS day_of_week,          -- 1 = Monday ... 7 = Sunday (ISO)
  date_format(date, 'EEEE')                               AS day_name,
  date_format(date, 'EEE')                                AS day_short_name,
  -- Flags
  weekday(date) >= 5                                      AS is_weekend,
  weekday(date) < 5                                       AS is_weekday,
  date = CAST(trunc(date, 'MM') AS DATE)                  AS is_month_start,
  date = last_day(date)                                   AS is_month_end,
  date = CAST(date_trunc('QUARTER', date) AS DATE)        AS is_quarter_start,
  date = CAST(add_months(date_trunc('QUARTER', date), 3) AS DATE) - 1 AS is_quarter_end,
  month(date) = 1 AND day(date) = 1                       AS is_year_start,
  month(date) = 12 AND day(date) = 31                     AS is_year_end,
  -- Audit
  current_timestamp()                                     AS insert_timestamp
FROM dates
ORDER BY date
""")

display(spark.table(target_table).limit(10))

In [0]:
# Validate row count, range, uniqueness and spot-check known dates
display(spark.sql(f"""
SELECT
  count(*)                                  AS row_count,
  datediff(DATE'{end_date}', DATE'{start_date}') + 1 AS expected_rows,
  count(DISTINCT datekey)                   AS distinct_datekeys,
  min(date)                                 AS min_date,
  max(date)                                 AS max_date,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM {target_table}
"""))

display(spark.sql(f"""
SELECT datekey, date, day_name, day_of_week, week, iso_year_week, quarter, is_weekend, is_quarter_end
FROM {target_table}
WHERE date IN (DATE'1940-01-01', DATE'2000-02-29', DATE'2026-10-28', DATE'2030-12-31')
ORDER BY date
"""))